# XAUUSD Real-Data Backtest Review

Notebook này dùng bộ dữ liệu nhiều khung trong `src/xauusd_ai/real_data`, chuẩn hoá dữ liệu nếu cần, đọc kết quả backtest spot thật đã chạy bằng Docker, và phân tích các khía cạnh trước khi chuyển sang paper trade.

## 1. Nạp thư viện và cấu hình đường dẫn
Thiết lập đường dẫn, phí, slippage, vốn ban đầu và tham số chạy thử.

## 2. Đọc dữ liệu đa khung từ `real_data`
Nạp các file `D1`, `H4`, `H1`, `M30`, `M15`, `M5`, `M1` vào dictionary chung.

## 3. Kiểm tra schema và chuẩn hoá dữ liệu OHLCV
Kiểm tra cột, kiểu dữ liệu, duplicate timestamp, missing candle và giá trị bất thường.

## 4. Đồng bộ timestamp và chuẩn bị dữ liệu đa timeframe
Align dữ liệu khung lớn xuống khung vào lệnh mà không leak tương lai.

In [ ]:
from __future__ import annotations

import json
import pickle
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import seaborn as sns
except ImportError:
    sns = None

try:
    import plotly.express as px
except ImportError:
    px = None

plt.style.use("ggplot")


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root")


PROJECT_ROOT = find_project_root(Path.cwd())
REAL_DATA_DIR = PROJECT_ROOT / "src" / "xauusd_ai" / "real_data"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
CONFIG_PATH = PROJECT_ROOT / "configs" / "settings.yaml"
FEE_BPS = 2.0
SLIPPAGE_BPS = 1.5
INITIAL_BALANCE = 10_000.0
RERUN_BACKTEST = False

TIMEFRAMES = ["D1", "H4", "H1", "M30", "M15", "M5", "M1"]
TIMEFRAME_FILES = {timeframe: REAL_DATA_DIR / f"XAUUSDm_{timeframe}.csv" for timeframe in TIMEFRAMES}

print(PROJECT_ROOT)
print(REAL_DATA_DIR)
print(OUTPUTS_DIR)

## 5. Tạo feature và regime cho sideway / biến động mạnh
Tạo các feature nền để đọc trạng thái thị trường: ATR, realized volatility, trend strength, range compression và regime sideway hoặc strong-volatility.

## 6. Chạy backtest spot trên dữ liệu thật
Notebook có thể gọi lại pipeline backtest trong Docker hoặc chỉ đọc các artifact đã được xuất sẵn.

## 7. Tính metrics và lưu lịch sử lệnh
Đọc `backtest_report.json`, `backtest_trades.csv` và `training_dataset.csv` để phân tích lại.

## 8. Vẽ equity curve và drawdown
Hiển thị đường vốn và underwater curve.

## 9. Phân tích phân phối trade
Xem phân phối lợi nhuận từng lệnh, holding surrogate và cấu trúc lời/lỗ.

In [ ]:
REQUIRED_COLUMNS = ["time", "open", "high", "low", "close"]
OPTIONAL_COLUMNS = ["tick_volume", "spread", "spread_points", "real_volume"]


def load_and_normalize_csv(csv_path: Path, timeframe: str) -> pd.DataFrame:
    frame = pd.read_csv(csv_path)
    frame = frame.rename(
        columns={
            "Datetime": "time",
            "Date": "time",
            "Open": "open",
            "High": "high",
            "Low": "low",
            "Close": "close",
            "Volume": "tick_volume",
            "TickVolume": "tick_volume",
            "Spread": "spread_points",
            "spread": "spread_points",
        }
    )
    missing = sorted(set(REQUIRED_COLUMNS) - set(frame.columns))
    if missing:
        raise ValueError(f"{csv_path.name} missing columns: {missing}")

    if "tick_volume" not in frame.columns:
        frame["tick_volume"] = 0.0
    if "spread_points" not in frame.columns:
        frame["spread_points"] = frame.get("spread", 0.0)

    numeric_columns = ["open", "high", "low", "close", "tick_volume", "spread_points"]
    for column in numeric_columns:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")

    frame["time"] = pd.to_datetime(frame["time"], utc=True, errors="coerce")
    frame = frame.dropna(subset=["time", "open", "high", "low", "close"]).copy()
    frame = frame.sort_values("time").drop_duplicates(subset=["time"]).reset_index(drop=True)
    frame = frame[(frame[["open", "high", "low", "close"]] > 0).all(axis=1)].copy()
    frame["timeframe"] = timeframe
    frame["tick_volume_delta"] = frame["tick_volume"].diff().fillna(0)
    frame["range"] = frame["high"] - frame["low"]
    frame["body"] = (frame["close"] - frame["open"]).abs()
    frame["volume_imbalance"] = (frame["body"] / frame["range"].replace(0, np.nan)).fillna(0)
    return frame


multi_tf = {timeframe: load_and_normalize_csv(path, timeframe) for timeframe, path in TIMEFRAME_FILES.items()}

schema_summary = pd.DataFrame(
    {
        timeframe: {
            "rows": len(frame),
            "start": frame["time"].min(),
            "end": frame["time"].max(),
            "duplicates": int(frame["time"].duplicated().sum()),
            "null_ohlc": int(frame[["open", "high", "low", "close"]].isna().sum().sum()),
        }
        for timeframe, frame in multi_tf.items()
    }
).T

schema_summary

In [ ]:
def compute_atr(frame: pd.DataFrame, period: int = 14) -> pd.Series:
    high_low = frame["high"] - frame["low"]
    high_close = (frame["high"] - frame["close"].shift()).abs()
    low_close = (frame["low"] - frame["close"].shift()).abs()
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    return tr.rolling(period).mean()


def compute_adx_like(frame: pd.DataFrame, period: int = 14) -> pd.Series:
    up_move = frame["high"].diff()
    down_move = -frame["low"].diff()
    plus_dm = up_move.where((up_move > down_move) & (up_move > 0), 0.0)
    minus_dm = down_move.where((down_move > up_move) & (down_move > 0), 0.0)
    atr = compute_atr(frame, period).replace(0, np.nan)
    plus_di = 100 * (plus_dm.rolling(period).sum() / atr)
    minus_di = 100 * (minus_dm.rolling(period).sum() / atr)
    dx = ((plus_di - minus_di).abs() / (plus_di + minus_di).replace(0, np.nan)) * 100
    return dx.rolling(period).mean().fillna(0)


execution_frame = multi_tf["M15"].copy()
execution_frame["atr"] = compute_atr(execution_frame).bfill()
execution_frame["atr_ratio"] = execution_frame["atr"] / execution_frame["close"]
execution_frame["realized_vol"] = execution_frame["close"].pct_change().rolling(24).std().fillna(0)
execution_frame["adx_like"] = compute_adx_like(execution_frame)
execution_frame["range_compression"] = (execution_frame["range"].rolling(12).mean() / execution_frame["range"].rolling(48).mean()).fillna(1)
execution_frame["trend_strength"] = (execution_frame["close"].rolling(12).mean() - execution_frame["close"].rolling(48).mean()).abs() / execution_frame["close"]
execution_frame["regime"] = np.select(
    [
        (execution_frame["atr_ratio"] < 0.0018) & (execution_frame["adx_like"] < 18),
        (execution_frame["atr_ratio"] > 0.0035) | (execution_frame["realized_vol"] > execution_frame["realized_vol"].quantile(0.75)),
    ],
    ["sideway", "strong_volatility"],
    default="normal",
)

for timeframe in ["D1", "H4", "H1", "M30", "M5", "M1"]:
    source = multi_tf[timeframe][["time", "close"]].copy()
    source[f"bias_{timeframe}"] = np.sign(source["close"].rolling(5).mean() - source["close"].rolling(20).mean()).fillna(0)
    execution_frame = pd.merge_asof(
        execution_frame.sort_values("time"),
        source[["time", f"bias_{timeframe}"]].sort_values("time"),
        on="time",
        direction="backward",
    )

execution_frame[[column for column in execution_frame.columns if column.startswith("bias_")]] = execution_frame[[column for column in execution_frame.columns if column.startswith("bias_")]].fillna(0)
execution_frame[["time", "close", "atr_ratio", "realized_vol", "adx_like", "regime"]].tail()

In [ ]:
if RERUN_BACKTEST:
    subprocess.run(
        [
            "docker",
            "compose",
            "run",
            "--rm",
            "trainer",
            "python",
            "-m",
            "xauusd_ai.main",
            "backtest",
            "--config",
            "configs/settings.yaml",
        ],
        cwd=PROJECT_ROOT,
        check=True,
    )

backtest_report = json.loads((OUTPUTS_DIR / "backtest_report.json").read_text(encoding="utf-8"))
trades = pd.read_csv(OUTPUTS_DIR / "backtest_trades.csv", parse_dates=["time"])
dataset = pd.read_csv(OUTPUTS_DIR / "training_dataset.csv", parse_dates=["time"])

trades["hour"] = trades["time"].dt.hour
trades["result_label"] = np.where(trades["is_win"], "win", "loss")
trades["cumulative_pnl"] = trades["pnl"].cumsum()
trades["equity"] = INITIAL_BALANCE + trades["cumulative_pnl"]
trades["running_max_equity"] = trades["equity"].cummax()
trades["underwater_pct"] = (trades["equity"] / trades["running_max_equity"] - 1.0) * 100

summary_metrics = pd.Series(
    {
        "selected_threshold": backtest_report["train_metrics"].get("selected_threshold"),
        "recall": backtest_report["train_metrics"].get("recall"),
        "precision": backtest_report["train_metrics"].get("precision"),
        "f1": backtest_report["train_metrics"].get("f1"),
        "roc_auc": backtest_report["train_metrics"].get("roc_auc"),
        "return_pct": backtest_report.get("return_pct"),
        "profit_factor": backtest_report.get("profit_factor"),
        "max_drawdown_pct": backtest_report.get("max_drawdown_pct"),
        "trades": backtest_report.get("trades"),
        "win_rate": backtest_report.get("win_rate"),
    }
)
summary_metrics

## 10. Phân tích win/loss theo giờ
Xem số lệnh, win rate, expectancy và tổng PnL theo giờ mở lệnh.

## 11. Phân tích hiệu suất theo regime
So sánh sideway, strong volatility và normal.

## 12. Tối ưu strategy bằng search và walk-forward
Thử grid search đơn giản theo threshold để nhìn trade-off recall, precision, số lệnh và drawdown.

## 13. Tối ưu labeling để tăng recall
Review ảnh hưởng của `label_horizon` và `min_return_threshold` đang dùng trong pipeline.

## 14. Hiệu chỉnh ngưỡng quyết định trước paper trade
Xem nhanh các candidate threshold tốt cho recall nhưng không phá precision quá mạnh.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0, 0].plot(trades["time"], trades["equity"], color="#2f4858", linewidth=2)
axes[0, 0].set_title("Equity Curve")
axes[0, 0].set_ylabel("Balance")

axes[0, 1].fill_between(trades["time"], trades["underwater_pct"], 0, color="#b55d3d", alpha=0.35)
axes[0, 1].set_title("Underwater Curve")
axes[0, 1].set_ylabel("Drawdown %")

axes[1, 0].hist(trades["pnl"], bins=40, color="#c49a3a", edgecolor="white")
axes[1, 0].set_title("Trade PnL Distribution")
axes[1, 0].set_xlabel("PnL")

side_pnl = trades.groupby("side")["pnl"].sum().sort_values()
axes[1, 1].bar(side_pnl.index, side_pnl.values, color=["#b22222", "#1f7a1f"][: len(side_pnl)])
axes[1, 1].set_title("PnL by Trade Side")
axes[1, 1].set_ylabel("PnL")

fig.autofmt_xdate()
fig.tight_layout()
plt.show()

hourly = trades.groupby("hour").agg(
    trades=("pnl", "size"),
    wins=("is_win", "sum"),
    total_pnl=("pnl", "sum"),
    avg_pnl=("pnl", "mean"),
)
hourly["win_rate"] = hourly["wins"] / hourly["trades"]
hourly

In [ ]:
dataset["time"] = pd.to_datetime(dataset["time"], utc=True)
regime_lookup = execution_frame[["time", "regime"]].copy().sort_values("time")
trades_with_regime = pd.merge_asof(
    trades.sort_values("time"),
    regime_lookup,
    on="time",
    direction="backward",
)

regime_stats = trades_with_regime.groupby("regime").agg(
    trades=("pnl", "size"),
    wins=("is_win", "sum"),
    total_pnl=("pnl", "sum"),
    avg_pnl=("pnl", "mean"),
    median_pnl=("pnl", "median"),
)
regime_stats["win_rate"] = regime_stats["wins"] / regime_stats["trades"]
regime_stats = regime_stats.sort_values("total_pnl", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(regime_stats.index, regime_stats["win_rate"], color="#4d7c8a")
axes[0].set_title("Win Rate by Regime")
axes[0].set_ylim(0, 1)
axes[1].bar(regime_stats.index, regime_stats["total_pnl"], color="#6b9a62")
axes[1].set_title("Total PnL by Regime")
axes[1].axhline(0, color="black", linewidth=1)
fig.tight_layout()
plt.show()

if sns is not None:
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=trades_with_regime, x="regime", y="pnl")
    plt.title("PnL Distribution by Regime")
    plt.axhline(0, color="black", linewidth=1)
    plt.show()

regime_stats

In [ ]:
with (OUTPUTS_DIR / "model_meta.json").open("r", encoding="utf-8") as file_handle:
    model_meta = json.load(file_handle)
with (OUTPUTS_DIR / "model.pkl").open("rb") as file_handle:
    model = pickle.load(file_handle)
with (OUTPUTS_DIR / "scaler.pkl").open("rb") as file_handle:
    scaler = pickle.load(file_handle)

feature_columns = [
    "daily_bias",
    "hourly_bias",
    "trend_alignment",
    "rsi",
    "macd_hist",
    "atr_ratio",
    "range_efficiency",
    "liquidity_sweep",
    "order_flow_proxy",
    "wyckoff_phase",
    "volatility_regime",
    "session_return",
    "tick_volume_zscore",
    "spread_points",
    "strategy_score",
]

scored = dataset.dropna(subset=feature_columns).copy()
scored["probability"] = model.predict_proba(scaler.transform(scored[feature_columns]))[:, 1]
scored_test = scored[scored["split"] == "test"].copy()

thresholds = np.arange(0.30, 0.81, 0.02)
threshold_rows = []
for threshold in thresholds:
    threshold_predictions = (scored_test["probability"] >= threshold).astype(int)
    true_positive = ((scored_test["target"] == 1) & (threshold_predictions == 1)).sum()
    false_positive = ((scored_test["target"] == 0) & (threshold_predictions == 1)).sum()
    false_negative = ((scored_test["target"] == 1) & (threshold_predictions == 0)).sum()
    precision = true_positive / max(true_positive + false_positive, 1)
    recall = true_positive / max(true_positive + false_negative, 1)
    trades_count = int(threshold_predictions.sum())
    threshold_rows.append(
        {
            "threshold": threshold,
            "precision": precision,
            "recall": recall,
            "trades": trades_count,
        }
    )

threshold_scan = pd.DataFrame(threshold_rows)

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(threshold_scan["threshold"], threshold_scan["recall"], label="Recall", color="#1f7a1f")
ax1.plot(threshold_scan["threshold"], threshold_scan["precision"], label="Precision", color="#b22222")
ax1.axvline(model_meta["decision_threshold"], color="#2f4858", linestyle="--", label="Selected")
ax1.set_title("Threshold Scan for Precision / Recall")
ax1.set_xlabel("Decision Threshold")
ax1.set_ylabel("Score")
ax1.legend()
plt.show()

threshold_scan.sort_values(["recall", "precision"], ascending=False).head(10)

## 15. Mô phỏng paper trade
Cell dưới đây tạo khung paper trade review từ các nến mới nhất trong M15 và không gửi lệnh thật.

## 16. Thiết lập điều kiện an toàn trước auto trade
Áp các rule như kill switch, daily drawdown limit, giới hạn số lệnh và checklist trước khi bật auto trade.

In [ ]:
paper_trade_view = execution_frame[["time", "close", "regime"]].tail(20).copy()
paper_trade_view["selected_threshold"] = model_meta["decision_threshold"]
paper_trade_view["paper_signal"] = np.where(scored_test["probability"].tail(len(paper_trade_view)).reset_index(drop=True) >= model_meta["decision_threshold"], "watch", "skip")
paper_trade_view

safety_rules = pd.DataFrame(
    [
        ("Kill switch", "Stop if daily drawdown <= -3%"),
        ("Trade limit", "Max 3 trades per day before auto-trade"),
        ("Latency check", "Reject if latest candle is stale"),
        ("Spread filter", "Reject if spread exceeds normal band"),
        ("News filter", "Block around high-impact events"),
        ("Dry-run gate", "Require paper-trade pass before auto-trade"),
    ],
    columns=["rule", "condition"],
)

print("Diagnostic summary")
print(f"Selected threshold: {model_meta['decision_threshold']:.2f}")
print(f"Recall: {backtest_report['train_metrics']['recall']:.3f}")
print(f"Profit factor: {backtest_report['profit_factor']:.3f}")
print(f"Max drawdown %: {backtest_report['max_drawdown_pct']:.2f}")
print("Assessment: suitable for backtest review and paper trade only, not ready for live auto-trading yet.")

safety_rules